# RelCheck — Visual Genome Recall Test

Tests whether RelCheck correctly **detects** naturally-occurring relational hallucinations
in BLIP-2 captions, using Visual Genome ground-truth relation annotations.

## How it works
1. Download VG relationships.json (~50 MB compressed)
2. Pick N_IMAGES images with >= MIN_VG_RELS diverse VG relations
3. Generate BLIP-2 captions for those images
4. Extract Llama triples from each caption
5. Match caption triples to VG ground truth — identify **contradictions** (ground truth hallucinations)
6. Run RelCheck VQA detection on each contradiction triple
7. Compute: **recall = hallucinations detected / total ground truth hallucinations**

## What this proves
Synthetic injection tests confirm recall on known fabrications. This confirms recall on **naturally-occurring** captioner hallucinations — a harder, more honest test.


In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
!pip install together Pillow requests -q

import os, json, re, time, random, requests, base64, zipfile, io
from pathlib import Path
from collections import Counter
from io import BytesIO
from PIL import Image
from together import Together
from google.colab import drive

# ── Config ──────────────────────────────────────────────────
TOGETHER_API_KEY = ''   # <-- paste your Together.ai key
N_IMAGES         = 20   # VG images to test
MIN_VG_RELS      = 3    # min VG relations per image (richer = better signal)
CAPTIONER        = 'qwen'   # 'qwen' | 'llava' — captioner to evaluate
SAVE_DIR         = '/content/drive/MyDrive/RelCheck_Data/vg_recall'
VG_DATA_PATH     = f'{SAVE_DIR}/relationships_sample.json'

LLM_MODEL    = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'
VLM_MODEL    = 'Qwen/Qwen3-VL-8B-Instruct'          # used for VQA verification
# Captioner models (Together.ai VLM API — no local GPU load needed)
_CAPTIONER_MODELS = {
    'qwen':  'Qwen/Qwen3-VL-8B-Instruct',
    'llava': 'llava-hf/llava-1.5-7b-hf',
}
CAPTION_MODEL = _CAPTIONER_MODELS[CAPTIONER]

drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['TOGETHER_API_KEY'] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f'  API failed: {e}')
                return None

def vlm_call(messages, max_tokens=10, retries=3):
    return llm_call(messages, model=VLM_MODEL, max_tokens=max_tokens, retries=retries)

print('Setup done.')


In [ ]:
# ============================================================
# Cell 2 — Download VG relationships + select candidate images
# ============================================================
# VG relationships.json: ~50 MB compressed.
# We only parse the first STREAM_LIMIT entries to keep things light.

# VG data moved from Stanford to UW (Ranjay Krishna's server).
# Try v1.2.0 first, fall back to v1.0.0.
VG_REL_URLS = [
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1_2.json.zip',
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1.json.zip',
]
# If you manually downloaded the zip, upload it to Drive and set this path.
# Leave as None to use automatic download.
LOCAL_ZIP_PATH = f'{SAVE_DIR}/relationships.json.zip'   # set to None to force download
STREAM_LIMIT = 5000   # parse first 5000 images from VG

SKIP_PREDS = {'has','have','of','is','are','was','were','with',
              'and','a','an','the','it','its','for','to','by'}

def clean_vg_name(name):
    name = name.lower().strip()
    for art in ['a ', 'an ', 'the ']:
        if name.startswith(art):
            name = name[len(art):]
    return name.strip()

def vg_url(image_id):
    if image_id <= 108249:
        return f'https://cs.stanford.edu/people/rak248/VG_100K/{image_id}.jpg'
    return f'https://cs.stanford.edu/people/rak248/VG_100K_2/{image_id}.jpg'

# Load or download VG data
if Path(VG_DATA_PATH).exists():
    with open(VG_DATA_PATH) as f:
        vg_data = json.load(f)
    print(f'Loaded {len(vg_data)} VG entries from cache')
else:
    buf = None
    # Check for manually uploaded zip on Drive first
    if LOCAL_ZIP_PATH and Path(LOCAL_ZIP_PATH).exists():
        print(f'Found local zip at {LOCAL_ZIP_PATH} — using that')
        with open(LOCAL_ZIP_PATH, 'rb') as zf:
            buf = io.BytesIO(zf.read())
    else:
        print('Downloading VG relationships (trying multiple URLs)...')
        for url in VG_REL_URLS:
            try:
                print(f'  Trying {url}')
                resp = requests.get(url, stream=True, timeout=30)
                resp.raise_for_status()
                buf = io.BytesIO()
                total = 0
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    buf.write(chunk)
                    total += len(chunk)
                    print(f'  {total/1e6:.1f} MB...', end='\r')
                print(f'  Downloaded {total/1e6:.1f} MB from {url}')
                break
            except Exception as e:
                print(f'  Failed: {e}')
                buf = None
    if buf is None:
        raise RuntimeError(
            'All VG download URLs failed.\n'
            'Please manually download relationships.json.zip from '
            'https://visualgenome.org/api/v0/api_home and upload to Drive at:\n'
            f'{VG_DATA_PATH.replace(".json", ".json.zip")}'
        )
    buf.seek(0)
    with zipfile.ZipFile(buf) as zf:
        # zip may contain relationships.json or relationships_v1_2.json etc.
        fname = next(n for n in zf.namelist() if n.endswith('.json'))
        with zf.open(fname) as jf:
            raw = json.loads(jf.read().decode('utf-8'))
            vg_data = raw[:STREAM_LIMIT]
    with open(VG_DATA_PATH, 'w') as f:
        json.dump(vg_data, f)
    print(f'Saved {len(vg_data)} entries')

# Score images: prefer many distinct, concrete predicates
candidates = []
for entry in vg_data:
    img_id = entry['image_id']
    rels   = entry.get('relationships', [])
    good   = [r for r in rels
              if r.get('predicate','').strip().lower() not in SKIP_PREDS
              and len(r.get('predicate','').split()) <= 4
              and r.get('subject',{}).get('name')
              and r.get('object',{}).get('name')]
    if len(good) < MIN_VG_RELS:
        continue
    preds = {r['predicate'].lower() for r in good}
    candidates.append((len(preds), img_id, good[:10]))

candidates.sort(reverse=True)
random.seed(42)
pool = candidates[:min(200, len(candidates))]
selected = random.sample(pool, min(N_IMAGES, len(pool)))
selected_images = {img_id: rels for _, img_id, rels in selected}

print(f'Selected {len(selected_images)} images')
ex_id = list(selected_images.keys())[0]
print(f'Example [{ex_id}]:')
for r in selected_images[ex_id][:3]:
    s = clean_vg_name(r['subject']['name'])
    p = r['predicate']
    o = clean_vg_name(r['object']['name'])
    print(f'  ({s}, {p}, {o})')


In [ ]:
# ============================================================
# Cell 3 — Download VG images + caption via Together.ai VLM
# ============================================================
# Captions are generated via Together.ai API (no local GPU needed).
# Uses a detailed describe prompt so we get rich relational captions
# — much more likely to surface detectable hallucinations than
# BLIP-2's terse 10-word outputs.

CAPTIONS_PATH = f'{SAVE_DIR}/{CAPTIONER}_captions.json'

DESCRIBE_PROMPT = (
    'Describe this image in detail. Include all objects, '
    'their spatial positions relative to each other, any actions '
    'or interactions taking place, and notable attributes like colors and sizes.'
)

def caption_image(pil_image, model=CAPTION_MODEL, max_tokens=300):
    '''Generate a detailed caption via Together.ai VLM API.'''
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return llm_call(
        [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': DESCRIBE_PROMPT},
        ]}],
        model=model, max_tokens=max_tokens,
    )

# Download VG images
pil_images = {}
print('Downloading VG images...')
for img_id in selected_images:
    try:
        r = requests.get(vg_url(img_id), timeout=15)
        r.raise_for_status()
        pil_images[img_id] = Image.open(BytesIO(r.content)).convert('RGB')
    except Exception as e:
        print(f'  [{img_id}] download failed: {e}')
print(f'Downloaded {len(pil_images)}/{len(selected_images)} images')

# Generate captions
if Path(CAPTIONS_PATH).exists():
    with open(CAPTIONS_PATH) as f:
        captions = {int(k): v for k, v in json.load(f).items()}
    print(f'Loaded {len(captions)} {CAPTIONER} captions from cache')
else:
    print(f'Generating captions with {CAPTIONER} ({CAPTION_MODEL})...')
    captions = {}
    for img_id, pil in pil_images.items():
        cap = caption_image(pil)
        if cap:
            captions[img_id] = cap
            words = len(cap.split())
            print(f'  [{img_id}] {words}w: {cap[:80]}...')
        else:
            print(f'  [{img_id}] captioning failed')
    with open(CAPTIONS_PATH, 'w') as f:
        json.dump({str(k): v for k, v in captions.items()}, f)
    print(f'Captioned {len(captions)} images')

avg_words = sum(len(c.split()) for c in captions.values()) / max(len(captions), 1)
print(f'Avg caption length: {avg_words:.0f} words ({CAPTIONER})')
if captions:
    print('Example caption:')
    ex_id = list(captions.keys())[0]
    print(f'  [{ex_id}] {captions[ex_id][:200]}')
else:
    print('WARNING: No captions generated — check TOGETHER_API_KEY and CAPTIONER model ID')


In [ ]:
# ============================================================
# Cell 4 — Extract caption triples + identify GT hallucinations
# ============================================================
# A caption triple is a GT hallucination if:
#   - Same (subject, object) pair exists in VG (entity match)
#   - But the relation in the caption CONTRADICTS the VG relation
# We use Llama to judge compatibility — handles paraphrases cleanly.

TRIPLE_PROMPT = '''Extract relational claims from this caption as JSON.
Caption: "{caption}"
Output a JSON array: [{"subject": "...", "relation": "...", "object": "..."}]
Only claims about how two entities relate. ONLY valid JSON, no explanation.'''

COMPAT_PROMPT = '''Two descriptions of the same pair in an image:
  Caption says: {s} {cap_rel} {o}
  Ground truth: {s} {vg_rel} {o}
Are these COMPATIBLE (both can be true) or CONTRADICTORY (cannot both be true)?
Examples:
  COMPATIBLE: 'man riding horse' + 'man on horse' (same thing, different wording)
  CONTRADICTORY: 'man left of table' + 'man right of table'
Answer ONLY: COMPATIBLE or CONTRADICTORY'''

def entity_overlap(a, b):
    STOP = {'a','an','the','of','in','on','at','is','its','some','with'}
    wa = {w for w in a.lower().split() if w not in STOP and len(w) >= 3}
    wb = {w for w in b.lower().split() if w not in STOP and len(w) >= 3}
    if not wa or not wb:
        return a.lower().strip() == b.lower().strip()
    for w in wa:
        for v in wb:
            if w == v or (len(w) >= 4 and len(v) >= 4 and w[:4] == v[:4]):
                return True
    return False

GT_PATH     = f'{SAVE_DIR}/gt_hallucinations_{CAPTIONER}.json'
gt_hallucs  = {}   # img_id -> [{subject, cap_rel, object, vg_rel, claim}]
cap_triples = {}   # img_id -> [extracted triples]

if Path(GT_PATH).exists():
    with open(GT_PATH) as f:
        saved = json.load(f)
    gt_hallucs  = {int(k): v for k, v in saved['hallucinations'].items()}
    cap_triples = {int(k): v for k, v in saved['cap_triples'].items()}
    print(f'Loaded from cache: {sum(len(v) for v in gt_hallucs.values())} GT hallucinations')
else:
    print('Extracting triples and matching against VG ground truth...')
    for img_id, caption in captions.items():
        vg_rels = selected_images.get(img_id, [])
        raw = llm_call([{'role': 'user', 'content': TRIPLE_PROMPT.format(caption=caption)}],
                       max_tokens=400)
        try:
            clean = re.sub(r'```[a-z]*\n?', '', raw or '').replace('```', '').strip()
            triples = json.loads(clean)
        except Exception:
            triples = []
        cap_triples[img_id] = triples

        hallucinations = []
        for ct in triples:
            cs = clean_vg_name(ct.get('subject', ''))
            cr = ct.get('relation', '').lower().strip()
            co = clean_vg_name(ct.get('object', ''))
            if not cs or not cr or not co:
                continue
            already_halluc = False   # don't double-count same caption triple
            for vr in vg_rels:
                if already_halluc:
                    break
                vs = clean_vg_name(vr['subject']['name'])
                vp = vr['predicate'].lower().strip()
                vo = clean_vg_name(vr['object']['name'])
                if not (entity_overlap(cs, vs) and entity_overlap(co, vo)):
                    continue
                # Same entity pair found in VG
                if cr == vp:
                    # Identical relation — definitely compatible; no need to check
                    # other VG relations for this pair (one clear match is enough).
                    already_halluc = False
                    break
                compat = (llm_call([{'role': 'user', 'content':
                    COMPAT_PROMPT.format(s=cs, cap_rel=cr, o=co, vg_rel=vp)}],
                    max_tokens=5) or '').upper()
                if 'COMPATIBLE' in compat:
                    # One compatible VG match → caption triple is not a hallucination.
                    already_halluc = False
                    break
                if 'CONTRADICTORY' in compat:
                    hallucinations.append({
                        'subject': cs, 'cap_rel': cr, 'object': co,
                        'vg_rel': vp, 'claim': f'{cs} {cr} {co}',
                    })
                    print(f'  [{img_id}] HALLUCINATION: ({cs}, {cr}, {co}) VG says "{vp}"')
                    already_halluc = True   # found one contradiction — done with this triple
        gt_hallucs[img_id] = hallucinations

    with open(GT_PATH, 'w') as f:
        json.dump({'hallucinations': {str(k): v for k, v in gt_hallucs.items()}, 'captioner': CAPTIONER,
                   'cap_triples':    {str(k): v for k, v in cap_triples.items()}}, f)

total_hall = sum(len(v) for v in gt_hallucs.values())
n_affected = sum(1 for v in gt_hallucs.values() if v)
print(f'\nGT hallucinations: {total_hall} across {n_affected}/{len(captions)} images')
if total_hall == 0:
    print('WARNING: 0 hallucinations found. Try N_IMAGES=50 or lowering MIN_VG_RELS.')


In [ ]:
# ============================================================
# Cell 5 — RelCheck VQA detection + recall
# ============================================================
# For each GT hallucination, run RelCheck's VQA verifier.
# We test detection only (no correction) — just: does RelCheck flag it INCORRECT?

RECALL_PATH = f'{SAVE_DIR}/recall_results_{CAPTIONER}.json'

def _img_b64(pil_image):
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()

def check_triple_vqa(subj, rel, obj_, pil_image, n_questions=3):
    '''
    Run YES/NO VQA for (subj, rel, obj) against the image.
    Returns (detected: bool, yes_v, no_v, total)
    detected=True if majority vote says NO (relation is INCORRECT).
    '''
    if pil_image is None:
        return False, 0, 0, 0
    b64 = _img_b64(pil_image)
    questions = [
        f'Is the {subj} {rel} the {obj_}? Answer YES or NO only.',
        f'Does the {subj} appear to be {rel} the {obj_}? YES or NO.',
        f'In this image, is there a {subj} that is {rel} a {obj_}? YES or NO.',
    ][:n_questions]
    yes_v = no_v = 0
    for q in questions:
        r = vlm_call([{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': q},
        ]}], max_tokens=5)
        if not r: continue
        rl = r.strip().lower()
        if 'yes' in rl and 'no' not in rl:
            yes_v += 1
        elif 'no' in rl:
            no_v += 1
    total = yes_v + no_v
    if total == 0:
        return False, 0, 0, 0
    yes_ratio = yes_v / total
    detected = yes_ratio < 0.50   # majority NO = flagged as INCORRECT
    return detected, yes_v, no_v, total

recall_results = []
total_hall = sum(len(v) for v in gt_hallucs.values())   # recompute in case Cell 4 was cached
print(f'Running VQA detection on {total_hall} GT hallucinations...\n')

for img_id, hallucs in gt_hallucs.items():
    if not hallucs:
        continue
    pil     = pil_images.get(img_id)
    caption = captions.get(img_id, '')
    for hall in hallucs:
        s, r, o = hall['subject'], hall['cap_rel'], hall['object']
        detected, yes_v, no_v, total = check_triple_vqa(s, r, o, pil)
        result = {
            'img_id': img_id, 'caption': caption,
            'subject': s, 'cap_rel': r, 'object': o,
            'vg_rel': hall['vg_rel'],
            'detected': detected,
            'votes': f'{yes_v}Y/{no_v}N/{total}T',
        }
        recall_results.append(result)
        status = 'DETECTED' if detected else 'MISSED   '
        print(f'  [{img_id}] {status} ({s}, {r}, {o}) VG="{hall["vg_rel"]}" [{yes_v}Y/{no_v}N]')

with open(RECALL_PATH, 'w') as f:
    json.dump(recall_results, f, indent=2)

# ── Metrics ─────────────────────────────────────────────────
n_total    = len(recall_results)
n_detected = sum(1 for r in recall_results if r['detected'])
recall     = n_detected / max(n_total, 1)

print('\n' + '='*60)
print('RELCHECK RECALL — VISUAL GENOME GROUND TRUTH')
print('='*60)
print(f'  GT hallucinations : {n_total}')
print(f'  Detected          : {n_detected}')
print(f'  Recall            : {recall:.1%}')
print()
print('MISSED cases (failure analysis):')
missed = [r for r in recall_results if not r['detected']]
if not missed:
    print('  None — perfect recall!')
else:
    for r in missed:
        print(f'  [{r["img_id"]}] ({r["subject"]}, {r["cap_rel"]}, {r["object"]}) '
              f'VG="{r["vg_rel"]}" [{r["votes"]}]')
print('='*60)
